# Paper 1 schema probe 2 (not a test)
Same burned days as probe 1 (TWSE/TPEx 2024-01-02 and 2024-01-03). Reads price, index and ex-rights layouts, and checks whether securities absent from T86 still traded. Writes `schema_probe2_results.zip`.

In [ ]:
import io, json, time, zipfile, datetime, hashlib, requests
import pandas as pd
OUT, LOG = {}, []
H = {"User-Agent": "Mozilla/5.0 (research schema probe)"}
def get(name, url, params=None):
    t = datetime.datetime.utcnow().isoformat() + "Z"
    try:
        r = requests.get(url, headers=H, params=params, timeout=60)
        LOG.append({"name": name, "url": r.url, "status": r.status_code, "bytes": len(r.content), "utc": t,
                    "sha256": hashlib.sha256(r.content).hexdigest()})
        OUT[name + ".raw"] = r.content
        time.sleep(3)
        return r
    except Exception as e:
        LOG.append({"name": name, "url": url, "error": repr(e), "utc": t}); return None
def tables_summary(name, j):
    s = {"keys": list(j.keys())}
    for k, v in j.items():
        if isinstance(v, list) and v and isinstance(v[0], dict) and "fields" in v[0]:
            s[k] = [{"title": t.get("title"), "fields": t.get("fields"), "rows": len(t.get("data", [])),
                     "head": t.get("data", [])[:3]} for t in v]
        elif k.startswith("fields") or k.startswith("data"):
            s[k] = v[:3] if isinstance(v, list) else v
    OUT[name + ".schema.json"] = json.dumps(s, ensure_ascii=False, indent=1).encode()
    return s

In [ ]:
# TWSE all-securities daily quotes and index table (MI_INDEX), burned dates
mi = {}
for d in ["20240102", "20240103"]:
    r = get(f"twse_MI_INDEX_{d}", "https://www.twse.com.tw/rwd/zh/afterTrading/MI_INDEX",
            {"date": d, "type": "ALLBUT0999", "response": "json"})
    if r is not None and r.ok:
        try: mi[d] = r.json(); tables_summary(f"twse_MI_INDEX_{d}", mi[d])
        except Exception as e: LOG.append({"name": f"twse_MI_INDEX_{d}", "parse_error": repr(e)})

In [ ]:
# TWSE ex-rights / ex-dividend calculation results over the burned window
r = get("twse_TWT49U_20240102_20240103", "https://www.twse.com.tw/rwd/zh/exRight/TWT49U",
        {"startDate": "20240102", "endDate": "20240103", "response": "json"})
if r is not None and r.ok:
    try: tables_summary("twse_TWT49U_20240102_20240103", r.json())
    except Exception as e: LOG.append({"name": "twse_TWT49U", "parse_error": repr(e)})

In [ ]:
# TPEx daily close quotes, burned dates (ROC year)
for d in ["113/01/02", "113/01/03"]:
    tag = d.replace("/", "")
    r = get(f"tpex_close_{tag}", "https://www.tpex.org.tw/web/stock/aftertrading/daily_close_quotes/stk_quote_result.php",
            {"l": "zh-tw", "o": "json", "d": d})
    if r is not None and r.ok:
        try: tables_summary(f"tpex_close_{tag}", r.json())
        except Exception as e: LOG.append({"name": f"tpex_close_{tag}", "parse_error": repr(e)})

In [ ]:
# Missing-versus-zero check (layout only): which traded four-digit stocks are absent from T86?
# Upload the probe-1 zip when prompted; only its T86 raw files are read.
check = {}
try:
    from google.colab import files
    up = files.upload()
    z = zipfile.ZipFile(io.BytesIO(list(up.values())[0]))
    for d in ["20240102", "20240103"]:
        t86 = json.loads(z.read(f"twse_T86_{d}.raw"))
        in_t86 = {row[0].strip() for row in t86["data"]}
        quote_tab = None
        for k, v in mi.get(d, {}).items():
            if isinstance(v, list):
                for t in v:
                    if isinstance(t, dict) and t.get("fields") and "證券代號" in t["fields"] and len(t.get("data", [])) > 500:
                        quote_tab = t
        if quote_tab is None: check[d] = "quote table not found"; continue
        f = quote_tab["fields"]; ic = f.index("證券代號"); iv = f.index("成交股數")
        rows = [(x[ic].strip(), int(x[iv].replace(",", "") or 0)) for x in quote_tab["data"]]
        stocks = [(c, v) for c, v in rows if len(c) == 4 and c.isdigit()]
        traded = [(c, v) for c, v in stocks if v > 0]
        absent = [(c, v) for c, v in traded if c not in in_t86]
        vols = sorted(v for _, v in traded); med = vols[len(vols) // 2] if vols else None
        check[d] = {"four_digit_stocks_in_quotes": len(stocks), "traded": len(traded), "absent_from_T86_but_traded": len(absent),
                    "median_volume_all_traded": med, "median_volume_absent": sorted(v for _, v in absent)[len(absent) // 2] if absent else None,
                    "in_T86_but_not_traded": len([c for c in in_t86 if len(c) == 4 and c.isdigit() and c not in {s for s, _ in traded}])}
except Exception as e:
    check["error"] = repr(e)
OUT["missing_vs_zero_check.json"] = json.dumps(check, indent=1).encode()
print(json.dumps(check, indent=1))

In [ ]:
OUT["log.json"] = json.dumps(LOG, indent=1).encode()
with zipfile.ZipFile("schema_probe2_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for k, v in OUT.items(): z.writestr(k, v)
for e in LOG: print(e.get("name"), e.get("status", e.get("error", e.get("parse_error"))))
try:
    from google.colab import files; files.download("schema_probe2_results.zip")
except Exception: print("Saved schema_probe2_results.zip")